In [ ]:
%matplotlib inline

import numpy

from scipy.special import wofz   # for Voigt function
from scipy.special import roots_legendre  # for Gaussian quadrature
from scipy.sparse import diags
from scipy.linalg import solve_banded

from numba import njit

import matplotlib.pyplot as plt

from matplotlib_inline.backend_inline import set_matplotlib_formats
set_matplotlib_formats('svg')

## Feautriers method

The RTE is transformed into a second-order differential equation of $P$:

$$
\begin{aligned}
\mu^2 \frac{d^2 P(\tau, \mu)}{d\tau^2} = P(\tau,\mu) - S(\tau) \,,
\end{aligned}
$$

where $P$ is coupled to the radiation field through
$$
\begin{aligned}
J_\nu (\tau) &= \sum_\mu w_\mu P(\tau, \mu)\,.
\end{aligned}
$$

In [ ]:
def quadrature(k=5):
    """
    Returns the nodes and weights form a Gaussian
    quadrature with k points. Rescaled to an interval
    from [0, 1].
    """
    nodes, weights = roots_legendre(k)
    return nodes/2 + 0.5, weights/2

In [ ]:
def Tmatrix(τ, μ, format=None):
    tau = τ / μ
    n = len(tau)
    Δτ = numpy.roll(tau, -1) - tau
    Δτm = numpy.roll(Δτ, 1)

    # TODO
    A = None # above diagonal
    B = None # diagonal
    C = None # below diagonal

    # Boundary conditions top
    B[0] = 2 / Δτ[0]**2 + 2 / Δτ[0] + 1
    C[0] = 2 / Δτ[0]**2
    # Boundary conditions bottom
    A[n-1] = 2 / (Δτ[n-2] * (Δτ[n-2] + 2))
    B[n-1] = (2 + 2*Δτ[n-2] + Δτ[n-2]**2) / (Δτ[n-2] * (Δτ[n-2] + 2))

    if format == 'banded':  # for use with scipy.linalg.solve_banded
        T = numpy.zeros((3, n))
        T[0, 1:] = -C[:-1]
        T[1] = B
        T[2, :-1] = -A[1:]
        return T
    elif format == 'sparse':  # for use with scipy.sparse.linalg.spsolve
        return diags([-A[1:], B, -C[:-1]], [-1, 0, 1], format='csc')
    else:  # for use with numpy.linalg.solv
        return diags([-A[1:], B, -C[:-1]], [-1, 0, 1]).toarray()

### Visualise $\mathbf{T}$

In [ ]:
tau = numpy.logspace(-8, 3, 50)
T = Tmatrix(tau, 1)

In [ ]:
fig, ax = plt.subplots()
ax.spy(T, markersize=2);

### Solve $\mathbf{T}\vec{P}=\vec{S}$ for $\vec{S}=1$ everywhere

In [ ]:
S = numpy.ones_like(tau)
P = numpy.linalg.solve(T, S)

In [ ]:
from scipy.sparse.linalg import spsolve
from scipy.linalg import solve_banded

In [ ]:
tau = numpy.logspace(-5, 3, 100)
S = numpy.ones_like(tau)
T_arr = Tmatrix(tau, 1)
T_sp = Tmatrix(tau, 1, format='sparse')
T_bd = Tmatrix(tau, 1, format='banded')

In [ ]:
%timeit spsolve(T_sp, S) # sparse

In [ ]:
%timeit solve_banded((1,1), T_bd, S) # banded

In [ ]:
%timeit numpy.linalg.solve(T_arr, S) # full matrix

### Obtain $\mathbf{\Lambda}$ operator from 5-point Gaussian quadrature

$$
\begin{aligned}
\mathbf{T}P &= S \\
J &= \mathbf{\Lambda}S = \sum_\mu w_\mu P_\mu = \sum_\mu w_\mu \mathbf{T}_\mu^{-1} S \\
 \mathbf{\Lambda} &=  \sum_\mu w_\mu \mathbf{T}_\mu^{-1}
\end{aligned}
$$

In [ ]:
tau = numpy.logspace(-8, 3, 50)

In [ ]:
def lambda_matrix(tau, k=5):
    n = len(tau)
    L = numpy.zeros((n, n))
    for mu, w in zip(*quadrature(k)):
        T = Tmatrix(tau, mu)
        L += w * numpy.linalg.inv(T)
    return L

In [ ]:
tau = numpy.logspace(-8, 3, 50)
Λ = lambda_matrix(tau, 5)
fig, ax = plt.subplots()
ax.imshow(Λ, origin='upper', vmax=0.2, cmap='gist_gray_r', interpolation='nearest')
ax.tick_params(top=True, labeltop=True, labelbottom=False);

### Write function `solve_cs_direct()` that takes as arguments 𝜏, 𝐵, and 𝜀 and computes 𝑆 and 𝐽 using a direct solution for the problem of coherent scattering 

$$
S = (\mathbb{1}-(1-\varepsilon)\mathbf{\Lambda})^{-1}[\varepsilon B],
$$

$$
J = \mathbf{\Lambda}S
$$

In [ ]:
N = 50
test = numpy.random.uniform(size=(N, N)) * 10
%timeit numpy.linalg.inv(test)

In [ ]:
N = 500
test = numpy.random.uniform(size=(N, N)) * 10
%timeit numpy.linalg.inv(test)

In [ ]:
def solve_direct(τ, B, ε):
    #
    # Create lambda matrix
    # Invert to get source function S
    # Find J from lamba and S
    #
    return S, J